# **Medicine Data Cleaning**

## 1. Importing Libraries and Loading Data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import sys
sys.path.append(str(Path.cwd().parent))

from scripts.paths import RAW_DIR, CLEANED_DIR

df = pd.read_csv(RAW_DIR / "07_pharmacy_master_dirty.csv")

## 2. Initial Data Inspection

In [29]:
df.head()

,Medicine_ID,Medicine_Name,Category,Unit_Cost,Stock_Qty,Supplier,Expiry_Date
0,M1000,Salbutamol 100mg,Gastro,1797.90,2946,MedPlus,2026-07-17
1,M1001,Paracetamol 500mg,Gastro,2494.03,4941,MedPlus,2026-06-24
2,M1002,Aspirin 250mg,Analgesic,79.56,1744,XYZ Distributors,2024-08-23
3,M1003,Paracetamol 40mg,Gastro,1142.31,4512,NaN,2025-04-05
4,M1004,Atorvastatin 250mg,Other,1785.86,1475,ABC Healthcare,2025-09-18


In [22]:
df.shape

(180, 7)

In [25]:
df.columns.tolist()

['Medicine_ID',
 'Medicine_Name',
 'Category',
 'Unit_Cost',
 'Stock_Qty',
 'Supplier',
 'Expiry_Date']

In [24]:
df.dtypes

Medicine_ID              object
Medicine_Name            object
Category                 object
Unit_Cost               float64
Stock_Qty                 Int64
Supplier                 object
Expiry_Date      datetime64[ns]
dtype: object

In [27]:
df.isna().sum()

Medicine_ID       0
Medicine_Name     0
Category          0
Unit_Cost         0
Stock_Qty         1
Supplier         34
Expiry_Date       0
dtype: int64

In [28]:
df.duplicated().sum()

0

## 3. Audit of Specific Columns

In [30]:
df["Medicine_ID"].duplicated().sum()

0

In [31]:
df["Medicine_ID"].str.fullmatch(r"M\d{4}").value_counts(dropna=False)

Medicine_ID
True    180
Name: count, dtype: int64

In [32]:
df["Medicine_Name"].value_counts().head(20)

Medicine_Name
Azithromycin 40mg     6
Pantoprazole          6
Insulin 500mg         6
Salbutamol 500mg      5
Azithromycin          5
Aspirin 500mg         5
Metformin             5
Metformin 40mg        5
Salbutamol 100mg      4
Paracetamol 100mg     4
Ondansetron 250mg     4
Azithromycin 100mg    4
Metformin 10mg        4
Aspirin 40mg          4
Metformin 500mg       4
Insulin 40mg          4
Metformin 100mg       4
Pantoprazole 10mg     4
Paracetamol 40mg      4
Ondansetron 10mg      3
Name: count, dtype: int64

In [ ]:
# Check for leading/trailing whitespace
(df["Medicine_Name"] != df["Medicine_Name"].str.strip()).sum()

32

In [33]:
df["Category"].value_counts(dropna=False)

Category
Diabetes       38
Analgesic      35
Respiratory    27
Antibiotic     23
Gastro         22
Cardiac        22
Other          13
Name: count, dtype: int64

In [34]:
df["Supplier"].value_counts(dropna=False)

Supplier
MedPlus             41
NaN                 34
ABC Healthcare      34
XYZ Distributors    29
Apollo              24
Local Pharma        18
Name: count, dtype: int64

In [35]:
df["Unit_Cost"].describe()

count     180.000000
mean     1164.474667
std       713.893476
min        37.700000
25%       543.322500
50%      1164.175000
75%      1733.957500
max      2498.830000
Name: Unit_Cost, dtype: float64

In [36]:
(df["Unit_Cost"] <= 0).sum()

0

In [37]:
df["Stock_Qty"].describe()

count          179.0
mean     2664.832402
std      1457.700215
min            119.0
25%           1477.0
50%           2724.0
75%           4072.0
max           4970.0
Name: Stock_Qty, dtype: Float64

In [38]:
(df["Stock_Qty"] < 0).sum()

0

In [39]:
df.loc[df["Stock_Qty"] < 0, "Stock_Qty"]

Series([], Name: Stock_Qty, dtype: Int64)

In [43]:
expiry_temp = pd.to_datetime(df["Expiry_Date"], errors="coerce")

In [40]:
expiry_temp.dtype

dtype('<M8[ns]')

In [41]:
expiry_temp.isna().sum()

0

In [42]:
expiry_temp.min(), expiry_temp.max()

(Timestamp('2023-08-30 00:00:00'), Timestamp('2029-04-23 00:00:00'))

In [44]:
# Expired dates are retained as valid historical/business information;
# this is an audit observation rather than a data-format error.
(expiry_temp < pd.Timestamp("2026-09-17")).sum()

94

## 4. Data Cleaning

### 4.1 Medicine_Name

In [10]:
medicine_name_cleaned = df["Medicine_Name"].str.strip()
medicine_name_cleaned.value_counts().head(20)

Medicine_Name
Azithromycin 40mg     6
Pantoprazole          6
Insulin 500mg         6
Salbutamol 500mg      5
Azithromycin          5
Aspirin 500mg         5
Metformin             5
Metformin 40mg        5
Salbutamol 100mg      4
Paracetamol 100mg     4
Ondansetron 250mg     4
Azithromycin 100mg    4
Metformin 10mg        4
Aspirin 40mg          4
Metformin 500mg       4
Insulin 40mg          4
Metformin 100mg       4
Pantoprazole 10mg     4
Paracetamol 40mg      4
Ondansetron 10mg      3
Name: count, dtype: int64

In [11]:
df["Medicine_Name"] = medicine_name_cleaned

### 4.2 Stock_Qty

In [47]:
stock_cleaned = df["Stock_Qty"].mask(df["Stock_Qty"] < 0)
stock_cleaned.isna().sum()

1

In [13]:
df["Stock_Qty"] = stock_cleaned.astype("Int64")

### 4.3 Expiry_Date

In [14]:
expiry_cleaned = pd.to_datetime(
    df["Expiry_Date"],
    errors="coerce"
)

expiry_cleaned.isna().sum()

0

In [15]:
df["Expiry_Date"] = expiry_cleaned

## 5. Final Validation

In [48]:
df.shape

(180, 7)

In [49]:
df.dtypes

Medicine_ID              object
Medicine_Name            object
Category                 object
Unit_Cost               float64
Stock_Qty                 Int64
Supplier                 object
Expiry_Date      datetime64[ns]
dtype: object

In [50]:
df.isna().sum()

Medicine_ID       0
Medicine_Name     0
Category          0
Unit_Cost         0
Stock_Qty         1
Supplier         34
Expiry_Date       0
dtype: int64

In [51]:
df.duplicated().sum()

0

In [52]:
df["Medicine_ID"].duplicated().sum()

0

In [53]:
df["Medicine_Name"].unique()

array(['Salbutamol 100mg', 'Paracetamol 500mg', 'Aspirin 250mg',
       'Paracetamol 40mg', 'Atorvastatin 250mg', 'Atorvastatin 500mg',
       'Ceftriaxone 500mg', 'Aspirin 500mg', 'Paracetamol 250mg',
       'Ceftriaxone 10mg', 'Insulin 250mg', 'Amoxicillin 100mg',
       'Pantoprazole', 'Ceftriaxone 250mg', 'Insulin 500mg',
       'Pantoprazole 10mg', 'Azithromycin 500mg', 'Ondansetron 10mg',
       'Azithromycin 40mg', 'Insulin 40mg', 'Salbutamol 40mg',
       'Aspirin 10mg', 'Ondansetron 500mg', 'Metformin 40mg',
       'Atorvastatin', 'Ondansetron 100mg', 'Metformin 100mg',
       'Salbutamol', 'Atorvastatin 100mg', 'Ibuprofen 100mg',
       'Azithromycin 250mg', 'Salbutamol 10mg', 'Pantoprazole 40mg',
       'Insulin 10mg', 'Aspirin 40mg', 'Azithromycin 10mg',
       'Pantoprazole 250mg', 'Metformin', 'Amoxicillin 250mg',
       'Azithromycin 100mg', 'Ceftriaxone 100mg', 'Azithromycin',
       'Ondansetron 250mg', 'Insulin', 'Ibuprofen', 'Pantoprazole 500mg',
       'Ibuprofen 50

In [54]:
df["Category"].unique()

array(['Gastro', 'Analgesic', 'Other', 'Diabetes', 'Cardiac',
       'Respiratory', 'Antibiotic'], dtype=object)

In [55]:
df["Supplier"].unique()

array(['MedPlus', 'XYZ Distributors', nan, 'ABC Healthcare', 'Apollo',
       'Local Pharma'], dtype=object)

In [56]:
(df["Medicine_Name"] != df["Medicine_Name"].str.strip()).sum()

0

In [57]:
(df["Stock_Qty"] < 0).sum()

0

In [58]:
(df["Unit_Cost"] <= 0).sum()

0

In [59]:
df["Expiry_Date"].isna().sum()

0

In [19]:
df["Expiry_Date"].min(), df["Expiry_Date"].max()

(Timestamp('2023-08-30 00:00:00'), Timestamp('2029-04-23 00:00:00'))

## 6. Export Cleaned Data

In [21]:
df.to_csv(CLEANED_DIR / "07_pharmacy_master_cleaned.csv", index=False)